# CNN Variant Training on Oxford Flowers 102

This notebook trains all CNN variants registered in `MODEL_REGISTRY` on the Oxford Flowers 102 dataset.
It compares model architectures in terms of parameter count, full-dataset accuracy, and few-shot learning performance.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import torch
import torch.nn as nn
import torch.optim as optim
import csv
import time
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

from models import MODEL_REGISTRY
from utils.data_loader import get_dataloaders, get_fewshot_dataloaders
from utils.set_seed import set_seed

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")
    print(f"CUDA version: {torch.version.cuda}")
print(f"Available models: {list(MODEL_REGISTRY.keys())}")

## Training Functions

In [ ]:
def train_one_epoch(model, dataloader, criterion, optimizer, device, scaler=None):
    """Train for one epoch. MixUp is handled in the dataloader collate_fn."""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for inputs, targets in dataloader:
        inputs, targets = inputs.to(device, non_blocking=True), targets.to(device, non_blocking=True)
        optimizer.zero_grad()
        
        with torch.amp.autocast('cuda', enabled=scaler is not None):
            logits, _ = model(inputs)
            loss = criterion(logits, targets)
        
        if scaler is not None:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()
        
        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(logits, 1)
        # Targets may be soft labels (batch, num_classes) from MixUp or hard labels (batch,)
        if targets.ndim == 1:
            correct += (predicted == targets).sum().item()
        else:
            correct += (predicted == targets.argmax(dim=1)).sum().item()
        total += inputs.size(0)
    return running_loss / total, correct / total

def validate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, targets in dataloader:
            inputs, targets = inputs.to(device, non_blocking=True), targets.to(device, non_blocking=True)
            with torch.amp.autocast('cuda', enabled=device.type == 'cuda'):
                logits, _ = model(inputs)
                loss = criterion(logits, targets)
            running_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(logits, 1)
            correct += (predicted == targets).sum().item()
            total += targets.size(0)
    return running_loss / total, correct / total

In [ ]:
def train_model(model_name, epochs=50, batch_size=32, lr=1e-3, k_shot=None, seed=69, mixup_alpha=0.0):
    """Train a model variant and return training history."""
    set_seed(seed)
    
    # Data (MixUp applied via collate_fn in dataloader)
    if k_shot:
        train_loader, val_loader, _ = get_fewshot_dataloaders(k_shot, batch_size=batch_size, mixup_alpha=mixup_alpha)
        print(f"Few-shot mode: {k_shot} images/class ({k_shot * 102} total)")
    else:
        train_loader, val_loader, _ = get_dataloaders(batch_size=batch_size, mixup_alpha=mixup_alpha)
        print(f"Full training set: 1020 images")
    
    # Model
    model = MODEL_REGISTRY[model_name](num_classes=102).to(device)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Model: {model_name} | Trainable params: {trainable:,}")
    
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    
    # Mixed precision scaler for GPU acceleration
    scaler = torch.amp.GradScaler('cuda') if device.type == 'cuda' else None
    if scaler:
        print("Mixed precision (AMP) enabled")
    if mixup_alpha > 0:
        print(f"MixUp enabled (alpha={mixup_alpha}, torchvision v2)")
    
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    best_val_acc = 0.0
    
    os.makedirs('../results/checkpoints', exist_ok=True)
    os.makedirs('../results/logs', exist_ok=True)
    shot_suffix = f'_{k_shot}shot' if k_shot else ''
    mixup_suffix = f'_mixup{mixup_alpha}' if mixup_alpha > 0 else ''
    checkpoint_path = f'../results/checkpoints/best_{model_name}{shot_suffix}{mixup_suffix}.pth'
    log_path = f'../results/logs/{model_name}{shot_suffix}{mixup_suffix}_training_log.csv'
    
    # CSV logging for evaluation notebook
    log_file = open(log_path, 'w', newline='')
    log_writer = csv.writer(log_file)
    log_writer.writerow(['epoch', 'train_loss', 'train_acc', 'val_loss', 'val_acc', 'lr'])
    
    pbar = tqdm(range(epochs), desc=f"Training {model_name}")
    for epoch in pbar:
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device, scaler)
        val_loss, val_acc = validate(model, val_loader, criterion, device)
        lr_val = scheduler.get_last_lr()[0]
        scheduler.step()
        
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        
        log_writer.writerow([epoch + 1, f'{train_loss:.4f}', f'{train_acc:.4f}',
                             f'{val_loss:.4f}', f'{val_acc:.4f}', f'{lr_val:.6f}'])
        log_file.flush()
        
        pbar.set_postfix({'train_acc': f'{train_acc:.3f}', 'val_acc': f'{val_acc:.3f}'})
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), checkpoint_path)
    
    log_file.close()
    print(f"Best val accuracy: {best_val_acc:.4f} | Saved to {checkpoint_path}")
    print(f"Training log: {log_path}")
    return history, best_val_acc

In [ ]:
def plot_training(histories, title="Training Comparison"):
    """Plot training curves for multiple models."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    for name, hist in histories.items():
        epochs = range(1, len(hist['train_acc']) + 1)
        axes[0].plot(epochs, hist['train_acc'], label=f'{name} (train)')
        axes[0].plot(epochs, hist['val_acc'], '--', label=f'{name} (val)')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].set_title('Accuracy')
    axes[0].legend(fontsize=8)
    axes[0].grid(True, alpha=0.3)
    
    for name, hist in histories.items():
        epochs = range(1, len(hist['train_loss']) + 1)
        axes[1].plot(epochs, hist['train_loss'], label=f'{name} (train)')
        axes[1].plot(epochs, hist['val_loss'], '--', label=f'{name} (val)')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].set_title('Loss')
    axes[1].legend(fontsize=8)
    axes[1].grid(True, alpha=0.3)
    
    fig.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.savefig(f'../results/logs/{title.replace(" ", "_").lower()}.png', dpi=150, bbox_inches='tight')
    plt.show()

## Model Parameter Comparison

In [ ]:
print(f"{'Model':<20} {'Trainable Params':>18} {'vs Baseline':>12}")
print("-" * 52)
baseline_params = None
for name, cls in MODEL_REGISTRY.items():
    model = cls(num_classes=102)
    params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    if name == 'baseline':
        baseline_params = params
        print(f"{name:<20} {params:>18,} {'':>12}")
    else:
        reduction = (1 - params / baseline_params) * 100
        sign = '-' if reduction > 0 else '+'
        print(f"{name:<20} {params:>18,} {sign}{abs(reduction):>10.1f}%")
    del model

## Train All Models (Full Training Set)

In [ ]:
# Train all CNN variants with identical hyperparameters
histories = {}
best_accs = {}

for model_name in MODEL_REGISTRY.keys():
    print(f"\n{'='*60}")
    print(f"Training: {model_name}")
    print(f"{'='*60}")
    hist, best_acc = train_model(model_name, epochs=50, batch_size=32, lr=1e-3)
    histories[model_name] = hist
    best_accs[model_name] = best_acc

In [ ]:
# Plot comparison
plot_training(histories, title="CNN Variants Comparison")

In [ ]:
# Summary table
print(f"\n{'Model':<20} {'Best Val Acc':>12}")
print("-" * 34)
for name, acc in sorted(best_accs.items(), key=lambda x: x[1], reverse=True):
    print(f"{name:<20} {acc:>12.4f}")

## Few-Shot Learning Experiments

In [ ]:
# Few-shot experiments: train each model with k=1,2,5,10 images per class
k_shots = [1, 2, 5, 10]
fewshot_results = {name: {} for name in MODEL_REGISTRY.keys()}

for k in k_shots:
    print(f"\n{'='*60}")
    print(f"Few-shot: k={k}")
    print(f"{'='*60}")
    for model_name in MODEL_REGISTRY.keys():
        print(f"\n--- {model_name} (k={k}) ---")
        _, best_acc = train_model(model_name, epochs=30, batch_size=32, lr=1e-3, k_shot=k)
        fewshot_results[model_name][k] = best_acc

In [ ]:
# Plot few-shot results
fig, ax = plt.subplots(figsize=(10, 6))
for name, results in fewshot_results.items():
    shots = sorted(results.keys())
    accs = [results[k] for k in shots]
    ax.plot(shots, accs, 'o-', label=name, linewidth=2, markersize=8)

ax.set_xlabel('Images per Class (k-shot)', fontsize=12)
ax.set_ylabel('Best Validation Accuracy', fontsize=12)
ax.set_title('Few-Shot Learning Comparison', fontsize=14)
ax.set_xticks(k_shots)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../results/logs/fewshot_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Few-shot results table
import pandas as pd
df = pd.DataFrame(fewshot_results).T
df.columns = [f'{k}-shot' for k in df.columns]
print(df.round(4).to_string())